# Apache Avro

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/avro.ipynb)

Apache Avro is a row-oriented binary serialization format with JSON schemas, a splittable container file, and well-defined schema evolution. It is the usual payload format for Kafka topics and CDC streams. The runnable cells use `fastavro`, in memory or in a temporary directory.

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"fastavro": "fastavro>=1.9"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

In [ ]:
import io
import json
import struct
import tempfile
from datetime import datetime, timedelta, timezone
from decimal import Decimal
from pathlib import Path

import fastavro
from fastavro import parse_schema, reader, schemaless_reader, schemaless_writer, writer

WORKDIR = Path(tempfile.mkdtemp(prefix="avro-demo-"))

ORDER_SCHEMA = {
    "type": "record",
    "name": "OrderEvent",
    "namespace": "com.example.orders",
    "doc": "One order lifecycle event emitted by the orders service.",
    "fields": [
        {"name": "order_id", "type": "string"},
        {"name": "customer_id", "type": "string"},
        {"name": "status", "type": {"type": "enum", "name": "OrderStatus",
                                    "symbols": ["CREATED", "PAID", "CANCELLED", "REFUNDED"]}},
        {"name": "amount", "type": {"type": "bytes", "logicalType": "decimal", "precision": 12, "scale": 2}},
        {"name": "currency", "type": "string", "default": "USD"},
        {"name": "event_time", "type": {"type": "long", "logicalType": "timestamp-millis"}},
    ],
}
parsed_order = parse_schema(ORDER_SCHEMA)
events = [
    {
        "order_id": f"O-{1000 + i}",
        "customer_id": f"C-{i % 50}",
        "status": ["CREATED", "PAID", "CANCELLED"][i % 3],
        "amount": Decimal(f"{(i * 37) % 10_000 / 100:.2f}"),
        "currency": "USD",
        "event_time": datetime(2026, 10, 4, 15, 3, 2, tzinfo=timezone.utc) + timedelta(seconds=i),
    }
    for i in range(5_000)
]
print("schema:", parsed_order["name"], "events:", len(events))

## What is Apache Avro?

Avro serializes data into a compact binary form described by a JSON schema, for files or messages. It was born in the Hadoop ecosystem and also defines an RPC protocol. The schema is the contract: a writer always serializes with a **writer schema**, and a reader resolves those bytes with its own **reader schema**.

Think of a record as a row of values where the types live in a separate contract, not in the bytes. The writer lays the fields down in schema order with no field names and no type tags. A `long` becomes a zig-zag varint that takes 1 byte for small values and up to 10 bytes for huge ones. A `string` becomes its byte length as a varint followed by UTF-8 bytes. The reader needs the writer schema to know where each value starts and ends.

In an Object Container File (`.avro`) the writer schema sits in the header, so the file is self-describing. On Kafka the schema usually lives in a schema registry and each message carries only a small schema id. Either way, a reader cannot decode Avro without the writer schema; that is true for JSON, not for Avro.

In [ ]:
record = events[1]
payload = io.BytesIO()
schemaless_writer(payload, parsed_order, record)
avro_bytes = payload.getvalue()
json_bytes = json.dumps(record, default=str).encode()
print(f"avro payload: {len(avro_bytes)} bytes  {avro_bytes.hex(' ')}")
print(f"json payload: {len(json_bytes)} bytes")

payload.seek(0)
assert schemaless_reader(payload, parsed_order) == record
assert len(avro_bytes) < len(json_bytes) / 3  # no field names, no type tags

## What does Avro offer?

+ Rich data structures: records, enums, arrays, maps, unions, fixed.
+ A compact, fast binary encoding with untagged data (no per-record field names or tags).
+ A splittable container file with the schema embedded and per-block compression.
+ Dynamic typing: a generic reader decodes any data with its schema, no code generation required.
+ Schema evolution through writer/reader schema resolution.
+ Language-neutral libraries (Java, Python, C, C++, C#, Go, Rust, and more) and an RPC protocol.

Field names are the identity. Avro has no numeric field ids like Protobuf, so a rename is a breaking change unless the reader declares an alias.

Inside an object container file the header holds the magic bytes `Obj` plus version 1, a metadata map, and a 16-byte random sync marker. The metadata map carries `avro.schema` (the writer schema as JSON) and `avro.codec`. After the header the file is a sequence of blocks. Each block is a record count, a byte size, the encoded records, and another copy of the sync marker. A reader can seek to any sync marker, so one file splits across parallel tasks.

A container file written and read back. The header holds `avro.schema` and `avro.codec`:

In [ ]:
orders_path = WORKDIR / "orders.avro"
with orders_path.open("wb") as out:
    writer(out, parsed_order, events, codec="deflate", sync_interval=16 * 1024)

with orders_path.open("rb") as src:
    avro_reader = reader(src)
    print("magic:", orders_path.read_bytes()[:4])
    print("codec:", avro_reader.metadata["avro.codec"])
    print("writer schema name:", avro_reader.writer_schema["name"])
    round_trip = list(avro_reader)

with orders_path.open("rb") as src:
    blocks = list(fastavro.block_reader(src))
print("blocks:", len(blocks), "records in first block:", blocks[0].num_records)

json_lines = sum(len(json.dumps(e, default=str)) + 1 for e in events)
print(f"avro file: {orders_path.stat().st_size:,} bytes vs json lines: {json_lines:,} bytes")

assert orders_path.read_bytes()[:4] == b"Obj\x01"
assert round_trip == events
assert len(blocks) > 1  # each block is a split point and a compression unit

Avro RPC is the second service: a protocol (`.avpr`) declares messages with request and response schemas, and a handshake exchanges protocol fingerprints before any payload moves. It is rare outside Hadoop-era systems; gRPC plus Protobuf is the common service stack today.

## How are Avro schemas declared?

A schema is one of four JSON shapes: a string naming a primitive or a previously declared named type (`"string"`, `"com.example.orders.OrderEvent"`), an object for a complex or logical type, a JSON array for a union (`["null", "string"]`), or an object with extra attributes. Records, enums, and fixed are **named** (full name = `namespace.name`); arrays, maps, and unions are anonymous. Declare each named type once and reference it by full name; a record may reference itself (lists, trees), and re-declaring the same full name fails validation.

Interview note: defaults are not optional metadata. A field with no `default` is required, and a union that includes `null` should normally declare `"default": null` so old readers of old data do not fail.

The **parsing canonical form** strips whitespace, `doc`, `aliases`, and defaults, and orders keys. A registry uses it (or its fingerprint) to detect that two schemas are the same. A `doc` edit does not produce a new canonical form:

In [ ]:
from fastavro.schema import fingerprint, to_parsing_canonical_form

edited = json.loads(json.dumps(ORDER_SCHEMA))
edited["doc"] = "Reworded documentation only."
canonical = to_parsing_canonical_form(ORDER_SCHEMA)
print(canonical[:120], "...")
print("fingerprint:", fingerprint(canonical, "CRC-64-AVRO"))

assert to_parsing_canonical_form(edited) == canonical
assert "doc" not in canonical

Ways to produce a schema in practice: hand-write the JSON for small, stable events; generate it from code (Avro Maven plugin, Java `SpecificData`/`ReflectData`, dataclass converters in Python); write Avro IDL and compile it; or let a CDC connector derive it from table DDL. Keep the schema in git next to the code and register the same bytes in the registry, so a rollback never replays a schema nobody can find.

```text
@namespace("com.example.orders")
protocol Orders {
  enum OrderStatus { CREATED, PAID, CANCELLED, REFUNDED }
  record OrderEvent {
    string order_id;
    string customer_id;
    OrderStatus status;
    decimal(12,2) amount;
    union { null, string } coupon_code = null;
  }
}
```

## What primitive and logical types does Avro support?

Primitives: `null` (0 bytes), `boolean` (1 byte), `int` and `long` (zig-zag varints), `float` (4 bytes) and `double` (8 bytes, little-endian), `bytes` and `string` (length varint plus payload).

Logical types annotate a primitive while the primitive still travels on the wire: `decimal` on `bytes` or `fixed` (with `precision` and `scale`), `date` on `int` (days since epoch), `time-millis` on `int`, `time-micros` on `long`, `timestamp-millis`/`timestamp-micros` on `long`, `uuid` on `string`, and `duration` on a 12-byte `fixed`. A reader that does not understand a logical type still reads the primitive but loses the meaning, so confirm consumer support.

In [ ]:
money = parse_schema({"type": "bytes", "logicalType": "decimal", "precision": 18, "scale": 2})
buf = io.BytesIO()
schemaless_writer(buf, money, Decimal("42.50"))
raw = buf.getvalue()
print("encoded decimal:", raw.hex(" "), "-> unscaled", int.from_bytes(raw[1:], "big", signed=True))

# A consumer whose schema says plain `bytes` (no logical type) gets raw bytes: the meaning is gone
buf.seek(0)
plain = schemaless_reader(buf, parse_schema("bytes"))
print("as plain bytes:", plain)

assert int.from_bytes(raw[1:], "big", signed=True) == 4250
assert plain == (4250).to_bytes(2, "big")

Common mistake: `timestamp-millis` on an `int`. A signed 32-bit value runs out about 24 days after the epoch, so timestamps need `long`. Another: money as `double`; moving to `decimal` later is a breaking type change.

## What complex types does Avro support?

```json
{
  "type": "record",
  "name": "Envelope",
  "namespace": "com.example.common",
  "fields": [
    {"name": "event_id", "type": "string"},
    {"name": "kind", "type": {"type": "enum", "name": "Kind", "symbols": ["ORDER", "PAYMENT", "REFUND"]}},
    {"name": "tags", "type": {"type": "array", "items": "string"}, "default": []},
    {"name": "attributes", "type": {"type": "map", "values": "string"}, "default": {}},
    {"name": "payload", "type": ["null", "com.example.orders.OrderEvent"], "default": null},
    {"name": "checksum", "type": {"type": "fixed", "name": "Md5", "size": 16}}
  ]
}
```

+ **Record**: named, ordered fields. **Enum**: named symbols; an enum `default` lets a reader resolve a symbol it does not know.
+ **Array** and **map**: maps always have string keys; use an array of records otherwise.
+ **Union**: an ordered list of types. No union inside a union and no two branches of the same unnamed type. A field default must match the **first** branch, so put `null` first for `"default": null`.
+ **Fixed**: named, sized bytes for checksums and keys.

In [ ]:
optional = parse_schema({"type": "record", "name": "A", "fields": [
    {"name": "coupon", "type": ["null", "string"], "default": None}]})
for value in [None, "SAVE10"]:
    buf = io.BytesIO()
    schemaless_writer(buf, optional, {"coupon": value})
    print(f"{value!r:10} -> {buf.getvalue().hex(' ')}")  # first byte is the union branch index

# Writing null into a field whose type has no null branch fails at write time
strict = parse_schema({"type": "record", "name": "C", "fields": [{"name": "coupon", "type": "string"}]})
try:
    schemaless_writer(io.BytesIO(), strict, {"coupon": None})
    null_rejected = False
except (TypeError, ValueError) as exc:
    null_rejected = True
    print("write failed:", type(exc).__name__)

assert null_rejected

## How do Avro, Protocol Buffers, and Thrift compare?

On the wire, Protobuf and Thrift write a tag or field id before each value, so they can skip unknown fields by number and renames are free. Avro writes values in schema order with no tags and relies on the writer schema plus name or alias matching, so renames need aliases. JSON writes field names in every record and needs no schema at all, at the cost of size.

+ Protobuf and Thrift use numeric field tags: renames are free but tags must never be reused. Both rely on an IDL and code generation.
+ Avro uses field names and supports both dynamic (generic) and generated (specific) records; renames need `aliases`.
+ Protobuf `oneof` maps to an Avro union; proto3 removed `required`, while Avro leans on defaults.

Rule of thumb: Avro for Kafka, Hadoop/Spark, and registry-driven evolution; Protobuf for gRPC service APIs; Thrift when you already run a Thrift service stack. Do not choose from a size benchmark alone and ignore the registry, tooling, and client libraries the team already runs.

## What are the advantages and disadvantages of Avro?

Advantages:
+ Small payloads: field names are paid once per file or per registered schema, not per record.
+ Splittable container files with per-block compression (`null`, `deflate`, `snappy`, `bzip2`, `xz`, `zstandard`).
+ Reading old data with a new schema (and vice versa) through resolution.
+ A machine-readable schema that can drive CI compatibility checks and documentation.

Disadvantages:
+ A schema is always required; bare Kafka payloads are undecodable without the registry.
+ Not human-readable; debugging needs tooling (`avro-tools tojson`, `fastavro` CLI).
+ Row-oriented: no column pruning, so analytical scans of a few columns are far cheaper in Parquet or ORC.
+ Reusing one schema for many event types leads to wide, optional-everything records.

Codec names recorded in `avro.codec` are `null`, `deflate`, `snappy`, `bzip2`, `xz`, and `zstandard`. Because each block is compressed on its own, blocks decode independently. Common mistake: gzipping an `.avro` file from the outside. The outer gzip is one opaque stream, so the file loses splittability. Set the codec on the writer instead.

In [ ]:
codec_sizes = {}
for codec in ["null", "deflate", "bzip2", "xz"]:
    path = WORKDIR / f"orders_{codec}.avro"
    with path.open("wb") as out:
        writer(out, parsed_order, events, codec=codec)
    codec_sizes[codec] = path.stat().st_size
    print(f"{codec:8} {codec_sizes[codec]:>9,} bytes")
assert codec_sizes["deflate"] < codec_sizes["null"]

```bash
java -jar avro-tools.jar getschema orders.avro
java -jar avro-tools.jar tojson --pretty orders.avro | head -3
```

## How does Avro serialization work?

Serialization turns a value into bytes using the schema. The Avro encoding rules: `int` and `long` are zig-zag encoded and then written as base-128 varints; `float` is 4 bytes and `double` is 8 bytes little-endian; `boolean` is one byte; `null` is zero bytes; strings and bytes are a length varint followed by the payload; arrays and maps are a sequence of blocks, each block a count then its items, closed by a zero-count block.

Common mistake: assuming varints make every number small. A value near 2^63 takes 10 bytes and a random 64-bit long averages about 9. Zig-zag keeps small positive and negative numbers cheap, which is why `-1` costs one byte. The exercises below implement the encoding by hand.

In [ ]:
small = parse_schema("long")
sizes = {}
for value in [0, -1, 63, 64, 8192, 2**63 - 1]:
    buf = io.BytesIO()
    schemaless_writer(buf, small, value)
    sizes[value] = len(buf.getvalue())
    print(f"{value:>20} -> {buf.getvalue().hex(' ')}")
assert sizes == {0: 1, -1: 1, 63: 1, 64: 2, 8192: 3, 2**63 - 1: 10}

## Which Avro APIs should you know?

+ Java: `Schema.Parser`, `GenericRecord`/`GenericData.Record`, `SpecificDatumWriter`/`SpecificDatumReader`, `DataFileWriter`/`DataFileReader`.
+ Python: `fastavro.parse_schema`, `writer`/`reader` (container files), `schemaless_writer`/`schemaless_reader` (bare payloads); the reference `avro` package (`avro.io.DatumWriter`, `avro.datafile.DataFileReader`).
+ Kafka: `confluent_kafka.schema_registry.avro.AvroSerializer` / `AvroDeserializer`.
+ Spark: `spark.read.format("avro")` and `from_avro`/`to_avro` via the `spark-avro` package.

Common mistake: writing with `schemaless_writer` and reading with `reader`. The bare payload has no header, so the reader finds no schema.

Producing to Kafka through a registry, and reading Avro in Spark:

```python
from confluent_kafka import Producer
from confluent_kafka.schema_registry import SchemaRegistryClient
from confluent_kafka.schema_registry.avro import AvroSerializer
from confluent_kafka.serialization import MessageField, SerializationContext

registry = SchemaRegistryClient({"url": "https://schema-registry.example.com"})
serializer = AvroSerializer(registry, order_schema_str)
producer = Producer({"bootstrap.servers": "kafka-1:9092,kafka-2:9092"})
producer.produce(
    topic="orders.events.v1",
    key="O-1001",
    value=serializer(event, SerializationContext("orders.events.v1", MessageField.VALUE)),
)
producer.flush()

# Spark (needs --packages org.apache.spark:spark-avro_2.12:3.5.1)
orders = spark.read.format("avro").load("s3://company-raw/orders/")
```

The Confluent wire format is one magic byte `0`, a 4-byte big-endian schema id, then the schemaless Avro payload. A simulation with an in-memory registry:

In [ ]:
registry: dict[int, dict] = {}


def register(schema: dict) -> int:
    canonical = to_parsing_canonical_form(schema)
    for schema_id, known in registry.items():
        if to_parsing_canonical_form(known) == canonical:
            return schema_id  # same canonical form: reuse the id
    registry[len(registry) + 1] = schema
    return len(registry)


def serialize(schema: dict, value: dict) -> bytes:
    buf = io.BytesIO()
    buf.write(struct.pack(">bI", 0, register(schema)))
    schemaless_writer(buf, parse_schema(schema), value)
    return buf.getvalue()


def deserialize(message: bytes, reader_schema: dict) -> dict:
    magic, schema_id = struct.unpack(">bI", message[:5])
    assert magic == 0
    writer_schema = parse_schema(registry[schema_id])
    return schemaless_reader(io.BytesIO(message[5:]), writer_schema, parse_schema(reader_schema))


message = serialize(ORDER_SCHEMA, events[0])
print("header:", message[:5].hex(" "), "payload bytes:", len(message) - 5)
assert deserialize(message, ORDER_SCHEMA) == events[0]
assert register(edited) == 1  # a doc-only edit reuses schema id 1

## How does Avro sort order work?

```json
{
  "type": "record",
  "name": "LeaderboardEntry",
  "fields": [
    {"name": "player_id", "type": "string"},
    {"name": "score", "type": "long", "order": "descending"},
    {"name": "updated_at", "type": {"type": "long", "logicalType": "timestamp-millis"}, "order": "ignore"}
  ]
}
```

Avro defines a total order per type so a reducer or external sort can compare records without knowing the writer's intent. Records compare field by field in declaration order, arrays and maps compare element-wise, and unions compare by branch index first. A field's `order` attribute (`ascending`, `descending`, `ignore`) changes comparison only, never the encoding.

Common mistake: expecting `order` to reorder stored bytes. It changes comparison only, and `ignore` still writes and reads the field.

## What is schema resolution?

Resolution reconciles the writer schema with the reader schema:
+ Record fields match by name, or by a reader-side alias.
+ A reader field the writer never wrote is filled from its `default`; without a default, resolution fails.
+ A writer field absent from the reader schema is skipped.
+ Widening promotions only: `int` to `long`/`float`/`double`, `long` to `float`/`double`, `float` to `double`, `string` and `bytes` interchangeably.
+ The writer's union branch must exist in the reader union (or promote into it).
+ An enum symbol unknown to the reader needs a reader enum `default`. Fixed sizes must match.

A default is applied only when the writer did not write the field at all. If old data contains `null` and the new reader expects `string`, no default saves the read.

In [ ]:
v1 = {"type": "record", "name": "Order", "namespace": "com.example.orders", "fields": [
    {"name": "order_id", "type": "string"},
    {"name": "status", "type": {"type": "enum", "name": "Status", "symbols": ["NEW", "PAID", "VOID"]}},
    {"name": "quantity", "type": "int"},
    {"name": "internal_note", "type": "string"},
]}
# v2: rename via reader alias, int -> long promotion, new field with a default,
# enum default for unknown symbols; internal_note is dropped and skipped on read
v2 = {"type": "record", "name": "Order", "namespace": "com.example.orders", "fields": [
    {"name": "order_id", "type": "string"},
    {"name": "state", "aliases": ["status"],
     "type": {"type": "enum", "name": "Status", "symbols": ["NEW", "PAID", "UNKNOWN"], "default": "UNKNOWN"}},
    {"name": "quantity", "type": "long"},
    {"name": "currency", "type": "string", "default": "USD"},
]}

old_rows = [
    {"order_id": "O-1", "status": "PAID", "quantity": 2, "internal_note": "x"},
    {"order_id": "O-2", "status": "VOID", "quantity": 1, "internal_note": "y"},
]
buf = io.BytesIO()
writer(buf, parse_schema(v1), old_rows)
buf.seek(0)
resolved = list(reader(buf, reader_schema=parse_schema(v2)))
for row in resolved:
    print(row)

assert resolved[0] == {"order_id": "O-1", "state": "PAID", "quantity": 2, "currency": "USD"}
assert resolved[1]["state"] == "UNKNOWN"  # unknown symbol falls back to the enum default

Without a default for the new field, the same read fails:

In [ ]:
v2_required = json.loads(json.dumps(v2))
v2_required["fields"][3].pop("default")
buf.seek(0)
try:
    list(reader(buf, reader_schema=parse_schema(v2_required)))
    resolution_failed = False
except fastavro.read.SchemaResolutionError as exc:
    resolution_failed = True
    print("resolution failed:", exc)
assert resolution_failed

## How does Avro schema evolution work?

| Change | Backward (new reader, old data) | Forward (old reader, new data) | Full |
|---|---|---|---|
| Add field with `default` | Yes | Yes | Yes |
| Add field without `default` | No | Yes | No |
| Remove field with `default` | Yes | Yes | Yes |
| Remove required field | Yes | No | No |
| Widen `int` to `long` | Yes | No | No |
| Rename field + add `aliases` | Yes | No | No |
| Change `string` to `int` | No | No | No |
| Reorder union branches | No | No | No |

Aliases rewrite the **writer's** schema using the **reader's** aliases, so an alias on the new schema makes a rename backward compatible. It is forward compatible only if the old reader also accepts the new name or the old field had a default.

Upgrade order follows the policy: BACKWARD means upgrade consumers first, FORWARD means producers first, FULL allows either. Make every new field optional with a default, give every rename a reader alias, and deprecate a removal for one release by giving the field a default first.

Worked example. v1 has `order_id` and `status`. v2 adds `currency` with default `USD` and renames `status` to `state` with `"aliases": ["status"]`. Old data decodes under v2 because `currency` falls back to its default and `state` matches the alias. New data decodes under v1 because the old reader skips `currency` and still finds `status`. That is FULL compatibility.

## What are Avro compatibility modes and how does a schema registry enforce them?

A registry stores versioned schemas per subject (usually `<topic>-value` and `<topic>-key`) and rejects a version that violates the subject's policy:
+ `BACKWARD` (default): the new schema reads data written with the previous one. Consumers upgrade first.
+ `FORWARD`: the previous schema reads data written with the new one. Producers upgrade first.
+ `FULL`: both directions; safest for rolling deployments.
+ `NONE`: no checks.
+ `*_TRANSITIVE` variants check against every registered version, not only the latest.

The mode is a policy, not a guarantee: a producer that bypasses the registry can still write incompatible bytes. Choose one subject naming strategy (TopicName, RecordName) and keep it, because switching resets a subject's compatibility history.

```bash
# Pin the policy for the subject, checking every historical version
curl -s -X PUT -H "Content-Type: application/vnd.schemaregistry.v1+json" \
  --data '{"compatibility": "BACKWARD_TRANSITIVE"}' \
  https://schema-registry.example.com/config/orders.events.v1-value

# CI gate: test a candidate schema without registering it
jq -Rs '{schema: .}' order_v2.avsc > payload.json
curl -s -X POST -H "Content-Type: application/vnd.schemaregistry.v1+json" \
  --data @payload.json \
  https://schema-registry.example.com/compatibility/subjects/orders.events.v1-value/versions/latest
```

Common mistake: setting BACKWARD and believing old consumers can read new data. BACKWARD protects new readers over old data. FORWARD protects old readers over new data. When both directions matter, use FULL.

## How do aliases work in Avro schema evolution?

```json
{
  "type": "record",
  "name": "OrderCreated",
  "aliases": ["OrderEvent"],
  "fields": [
    {"name": "order_id", "type": "string"},
    {"name": "gross_amount", "type": "double", "aliases": ["amount"]}
  ]
}
```

An alias is a reader-side name. During resolution the reader's aliases rewrite the writer's schema, so `gross_amount` with `"aliases": ["amount"]` reads a writer field named `amount`. Aliases also work on records, enums, and fixed types, and may be namespace-qualified. Adding an alias to the new reader makes a rename backward compatible. The rename is not forward compatible, because the old reader has no alias for the new name, unless the new field also has a default.

Common mistake: putting the alias on the writer schema or on the new field name. The alias belongs on the side that reads, and it points back at the old name.

## What are common Avro schema evolution pitfalls?

+ Adding a required field (no default) breaks backward compatibility for every existing reader.
+ Writing `null` into a union that does not declare `null` fails at serialization time, not at read time.
+ Removing `null` from `["null", "string"]` after data was written with nulls makes old data unreadable.
+ Using a bare `double` for money and later wanting `decimal` is a breaking type change; design money as decimal from day one.
+ Putting logically distinct events in one schema produces dozens of optional fields; prefer one topic/schema per event type with a shared envelope.
+ Relying on default `double`/`float` for IDs loses precision above 2^53.
+ Forgetting that a schema registry subject per topic means a rename of the record also changes the resolved schema everywhere.
+ Using `doc` or field descriptions as the only documentation; they are stripped from the canonical form and not enforced.

More traps: changing a decimal's `scale` or `precision` changes the value; removing an enum symbol makes old data unreadable; moving a record to a new namespace changes its full name; a union default that does not equal the first branch is rejected at parse time; and changing a field's type from `string` to `long` is not a legal promotion.

## How does SASL relate to Avro?

Avro RPC can run over SASL for authentication (`PLAIN`, `DIGEST-MD5`, `GSSAPI`/Kerberos, `ANONYMOUS` for trusted networks only). In modern Kafka pipelines SASL is configured on the Kafka client and broker, not in Avro. SASL authenticates the connection, not the data contract.

```yaml
# Kafka Connect worker: SASL_SSL plus Avro serialization via the registry
security.protocol: SASL_SSL
sasl.mechanism: SCRAM-SHA-512
value.converter: io.confluent.connect.avro.AvroConverter
value.converter.schema.registry.url: https://schema-registry.example.com
value.converter.basic.auth.credentials.source: USER_INFO
```

## Why is Avro useful in streaming and CDC pipelines?

Avro is compact, schema-based, and well suited for record-oriented data. It is commonly used with Kafka because producers and consumers can exchange small binary messages while a schema registry manages compatibility. This helps pipelines evolve without every consumer breaking when a new optional field is added.

In CDC, Debezium emits a well-known envelope (`before`, `after`, `op`, `source`) and its `AvroConverter` registers it in the registry:

```json
{
  "name": "orders-avro-connector",
  "config": {
    "connector.class": "io.debezium.connector.postgresql.PostgresConnector",
    "table.include.list": "public.orders,public.order_items",
    "plugin.name": "pgoutput",
    "topic.prefix": "cdc",
    "key.converter": "io.confluent.connect.avro.AvroConverter",
    "value.converter": "io.confluent.connect.avro.AvroConverter",
    "value.converter.schema.registry.url": "https://schema-registry.example.com",
    "decimal.handling.mode": "precise",
    "tombstones.on.delete": "true"
  }
}
```

Common mistake: writing with a schema that was never registered, or reusing a serializer that cached a stale schema. The id in the message then points at the wrong contract, and consumers decode the wrong fields.

## When should you use Avro?

Choose Avro when:
+ Kafka is the backbone and a schema registry is available.
+ Row-oriented files matter more than columnar scan performance (Avro beats Parquet for full-row streaming and write-heavy landing zones).
+ You need a self-describing container file for archival or handoff.

Do not choose Avro when:
+ The workload is analytical and scan-heavy by a few columns: Parquet or ORC will be much faster and smaller.
+ Humans need to read the payload directly.
+ You need random access or column pruning.

| Format | Layout | Schema | Best for | Weak spot |
|---|---|---|---|---|
| Avro | row, binary | writer schema or registry id | Kafka events, CDC, whole-row ETL | analytical column scans |
| Parquet | column, binary | embedded footer | lakehouse analytics, large scans | per-message streaming |
| ORC | column, binary | embedded footer | Hive-style analytics, ACID tables | ecosystem outside Hive |
| Protobuf | row, tagged | `.proto` or registry | gRPC APIs, polyglot services | needs code generation |
| JSON | row, text | none or JSON Schema | APIs, debugging, small volumes | size, no strict typing |

Common mistake: choosing one format for the whole platform. The usual answer is Avro on the stream and Parquet in the lake, with JSON kept at the edges for debugging.

## How does Avro compare with Hadoop Writable and Java serialization?

+ Java serialization writes class names, field metadata, and object graph information with every stream, which is large and not portable.
+ Hadoop `Writable` is compact and reusable (`write`/`readFields` into the same object), but it is Java-only and tightly coupled to a specific class.
+ Avro goes one step further: it is compact, cross-language, and schema-governed, so Hive, Spark, and Python can all read the same files.

`Writable` and SequenceFiles have only a Java API, which is why cross-language pipelines moved to Avro or Parquet. Avro does not replace `Writable` for shuffle: the shuffle format is internal to the engine, while Avro is for interchange and storage between systems and languages.

## How do you design Avro schemas and keep them compatible?

Make every field after the first release optional with a default. Declare an optional field as `["null", T]` with `"default": null`. Use a plain type only when the field is truly required for every event. Prefer a record over a union of records: unions grow combinatorially and every consumer must handle every branch. Use `aliases` for renames. Keep names and namespaces stable, because a namespace change renames the type.

v1:

```json
{
  "type": "record",
  "name": "Order",
  "namespace": "com.example.orders",
  "fields": [
    {"name": "order_id", "type": "string"},
    {"name": "status", "type": "string"}
  ]
}
```

v2 adds a defaulted field, an optional field, and a renamed field with an alias:

```json
{
  "type": "record",
  "name": "Order",
  "namespace": "com.example.orders",
  "fields": [
    {"name": "order_id", "type": "string"},
    {"name": "state", "type": "string", "aliases": ["status"]},
    {"name": "currency", "type": "string", "default": "USD"},
    {"name": "coupon", "type": ["null", "string"], "default": null}
  ]
}
```

Common mistake: making a field required because it is always present today. Messages already in the log have no value for it, so every new reader fails on old data unless the field has a default.

Storage choices: `zstandard` usually gives the best ratio per unit of CPU, `snappy` is fast with a modest ratio, and `deflate` is universal. `sync_interval` sets how often the 16-byte sync marker repeats: smaller intervals add seek points and per-block overhead, while huge blocks reduce read parallelism. Avro's JSON encoding is for debugging only; it is several times larger.

## Real world use cases for Apache Avro

### 1. E-commerce order events on Kafka

The checkout service publishes an `OrderEvent` per state change to five consuming teams (40M events/day, peak 3,000/s, ~120 bytes each). The producer uses `AvroSerializer`, the subject `orders.events.v1-value` is `BACKWARD`, and a Kafka Connect S3 sink writes container files to the raw zone. The registry blocks a producer from shipping a required field that would break consumers, and splittable files keep the nightly Spark job parallel.

### 2. Fintech ledger CDC into a lakehouse

Debezium reads the Postgres WAL and emits the Avro envelope; Spark Structured Streaming decodes with `from_avro`, deduplicates on `ledger_id` ordered by `source.lsn`, and merges into Iceberg. `decimal(18,2)` preserves exact money, `source.lsn` makes replays deterministic, and the raw Avro topic is the immutable audit trail.

### 3. Multi-tenant SaaS telemetry

A shared `EventEnvelope` carries `tenant_id`, `event_name`, `occurred_at`, and a `map<string,string>` of tenant properties, decoded by one generic consumer. One stable core schema avoids 2,000 registered schemas on the hot path, while per-tenant subjects govern optional typed extensions.

### 4. GDPR-safe events with tokenized PII

PII is stored in a vault and replaced with a `uuid`-typed token before serialization. Deleting the vault mapping makes lake rows meaningless, and the registry review stops a producer from quietly adding an `email` field.

### 5. Cross-language ML feature logging

Online services (Go, Java) write container files of `FeatureLog` records (`map<string,double>` features plus prediction); offline Spark and Python read them months later. Self-describing files decode without the writing service, and storing the schema with the model artifact makes training data reproducible.

## Avro exercises

These exercises use only the standard library to model Avro encoding and compatibility. Sizes are spec-based; the point is knowing which rule to apply and which direction a change moves compatibility.

### Try it 1. Compute the varint byte size of an integer

**Task.** Compute the Avro encoded size for `0`, `-1`, `63`, `64`, `8191`, `8192`, and `2**31 - 1`.

**Hint.** Avro zig-zags the value first: `zigzag(n) = (n << 1) ^ (n >> 63)`. The byte count is `ceil(bit_length / 7)`, with a minimum of one byte.

**Expected result.** 0 and -1 take 1 byte, 63 takes 1, 64 takes 2, 8191 takes 2, 8192 takes 3, and 2**31 - 1 takes 5.

In [ ]:
def zigzag(n):
    return (n << 1) ^ (n >> 63)


def varint_size(n):
    z = zigzag(n)
    return max(1, (z.bit_length() + 6) // 7)


sizes = {n: varint_size(n) for n in [0, -1, 63, 64, 8191, 8192, 2**31 - 1]}
for value, size in sizes.items():
    print(f"{value:>12} -> {size} byte(s)")

assert sizes[0] == 1
assert sizes[-1] == 1
assert sizes[63] == 1
assert sizes[64] == 2
assert sizes[8191] == 2
assert sizes[8192] == 3
assert sizes[2**31 - 1] == 5

### Try it 2. Decide if a schema change is BACKWARD or FORWARD compatible

**Task.** Given an old and a new record schema dict, decide whether the change is `FULL`, `BACKWARD`, `FORWARD`, or `NEITHER`. Test three changes: adding an optional field, adding a required field, and dropping a required field.

**Hint.** BACKWARD means the new schema reads old data, so a field added in the new schema needs a `default`. FORWARD means the old schema reads new data, so a field removed from the new schema needs a `default` in the old schema.

**Expected result.** Adding an optional field is FULL, adding a required field is FORWARD, and dropping a required field is BACKWARD.

In [ ]:
def fields(schema):
    return {f["name"]: f for f in schema["fields"]}


def backward(old, new):
    old_fields = fields(old)
    return all(name in old_fields or "default" in field
               for name, field in fields(new).items())


def forward(old, new):
    new_fields = fields(new)
    return all(name in new_fields or "default" in field
               for name, field in fields(old).items())


def verdict(old, new):
    b, f = backward(old, new), forward(old, new)
    if b and f:
        return "FULL"
    if b:
        return "BACKWARD"
    if f:
        return "FORWARD"
    return "NEITHER"


base = {"type": "record", "name": "Order", "fields": [
    {"name": "order_id", "type": "string"},
    {"name": "status", "type": "string"}]}
changes = {
    "add optional": {"type": "record", "name": "Order", "fields": [
        {"name": "order_id", "type": "string"},
        {"name": "status", "type": "string"},
        {"name": "currency", "type": "string", "default": "USD"}]},
    "add required": {"type": "record", "name": "Order", "fields": [
        {"name": "order_id", "type": "string"},
        {"name": "status", "type": "string"},
        {"name": "currency", "type": "string"}]},
    "drop required": {"type": "record", "name": "Order", "fields": [
        {"name": "order_id", "type": "string"}]},
}

for label, new in changes.items():
    print(f"{label:14} -> {verdict(base, new)}")

assert verdict(base, changes["add optional"]) == "FULL"
assert verdict(base, changes["add required"]) == "FORWARD"
assert verdict(base, changes["drop required"]) == "BACKWARD"

### Try it 3. Compute a record size from its field types

**Task.** For the record `{"order_id": "O-1001", "quantity": 3, "price": 19.99, "active": True}`, compute each field's encoded size and the total.

**Hint.** `string` is a length varint plus the UTF-8 bytes; `int` and `long` are varints; `double` is 8 bytes; `boolean` is 1 byte; `null` is 0 bytes.

**Expected result.** order_id is 7 bytes, quantity is 1, price is 8, active is 1, so the total is 17 bytes.

In [ ]:
def zigzag(n):
    return (n << 1) ^ (n >> 63)


def varint_size(n):
    z = zigzag(n)
    return max(1, (z.bit_length() + 6) // 7)


def size_of(type_name, value):
    if type_name == "null":
        return 0
    if type_name == "boolean":
        return 1
    if type_name in ("int", "long"):
        return varint_size(value)
    if type_name == "float":
        return 4
    if type_name == "double":
        return 8
    if type_name == "string":
        raw = value.encode()
        return varint_size(len(raw)) + len(raw)
    raise ValueError(type_name)


record = {"order_id": ("string", "O-1001"), "quantity": ("int", 3),
          "price": ("double", 19.99), "active": ("boolean", True)}
total = 0
for name, (type_name, value) in record.items():
    n = size_of(type_name, value)
    total += n
    print(f"{name:9} {type_name:8} {n:>3} bytes")
print("record payload:", total, "bytes")

assert size_of("string", "O-1001") == 7
assert size_of("int", 3) == 1
assert size_of("double", 19.99) == 8
assert total == 17

### Try it 4. Validate a schema dict structurally

**Task.** Write a validator that walks a schema and reports missing required keys. It must handle `record`, `enum`, `array`, `map`, and `fixed`, plus unions. Run it on a valid schema and on one that is missing a record name and a field type.

**Hint.** A record needs `name` and a `fields` list; every field needs `name` and `type`; an enum needs `name` and `symbols`; an array needs `items`; a map needs `values`; a fixed needs `name` and `size`; a union cannot contain another union.

**Expected result.** The valid schema reports no errors. The broken schema reports exactly two: the record has no name, and its first field has no type.

In [ ]:
def validate(schema, path="$"):
    errors = []
    if isinstance(schema, list):
        if not schema:
            errors.append(f"{path}: union must have at least one branch")
        if any(isinstance(branch, list) for branch in schema):
            errors.append(f"{path}: union cannot contain another union")
        for i, branch in enumerate(schema):
            errors.extend(validate(branch, f"{path}[{i}]"))
        return errors
    if isinstance(schema, str):
        return errors
    if not isinstance(schema, dict):
        return [f"{path}: schema must be a string, object, or array"]
    kind = schema.get("type")
    if kind == "record":
        if "name" not in schema:
            errors.append(f"{path}: record needs a name")
        if not isinstance(schema.get("fields"), list):
            errors.append(f"{path}: record needs a fields array")
        else:
            for i, field in enumerate(schema["fields"]):
                if "name" not in field or "type" not in field:
                    errors.append(f"{path}.fields[{i}]: field needs name and type")
    elif kind == "enum":
        if "name" not in schema or not isinstance(schema.get("symbols"), list):
            errors.append(f"{path}: enum needs a name and symbols")
    elif kind == "array":
        if "items" not in schema:
            errors.append(f"{path}: array needs items")
    elif kind == "map":
        if "values" not in schema:
            errors.append(f"{path}: map needs values")
    elif kind == "fixed":
        if "name" not in schema or "size" not in schema:
            errors.append(f"{path}: fixed needs a name and size")
    return errors


good = {"type": "record", "name": "Order", "fields": [
    {"name": "order_id", "type": "string"},
    {"name": "tags", "type": {"type": "array", "items": "string"}},
    {"name": "props", "type": {"type": "map", "values": "string"}},
    {"name": "state", "type": {"type": "enum", "name": "State", "symbols": ["NEW", "PAID"]}},
    {"name": "checksum", "type": {"type": "fixed", "name": "Md5", "size": 16}},
]}
bad = {"type": "record", "fields": [{"name": "order_id"}]}

print("good:", validate(good))
print("bad: ", validate(bad))

assert validate(good) == []
assert len(validate(bad)) == 2

### Try it 5. Detect a breaking change: a dropped field without a default

**Task.** Given an old and a new schema, list the fields that were dropped from the new schema without a default. Those fields break FORWARD compatibility.

**Hint.** Collect the new field names, then return every old field that is absent from the new schema and has no `default` key.

**Expected result.** Only `status` is reported; `currency` is dropped but has a default, so it is safe.

In [ ]:
def dropped_required(old, new):
    new_names = {f["name"] for f in new["fields"]}
    return [f["name"] for f in old["fields"]
            if f["name"] not in new_names and "default" not in f]


old = {"type": "record", "name": "Order", "fields": [
    {"name": "order_id", "type": "string"},
    {"name": "status", "type": "string"},
    {"name": "currency", "type": "string", "default": "USD"}]}
new = {"type": "record", "name": "Order", "fields": [
    {"name": "order_id", "type": "string"}]}

breaks = dropped_required(old, new)
print("dropped without default:", breaks)

assert breaks == ["status"]

### Try it 6. Compare storage for row versus column layout

**Task.** A table has columns `id` (8 bytes), `user` (16), `payload` (64), and `flags` (4), with 1,000,000 rows. A query touches a column-access profile of id 100%, user 50%, payload 5%, flags 10%. Compute the bytes read by a row layout and by a column layout.

**Hint.** A row layout reads the full width of every row. A column layout reads `column_bytes * rows * access_fraction` for each accessed column.

**Expected result.** The row layout reads 92,000,000 bytes and the column layout about 19,600,000, which is 0.213 of the row bytes.

In [ ]:
COLUMNS = {"id": 8, "user": 16, "payload": 64, "flags": 4}
ROWS = 1_000_000
profile = {"id": 1.0, "user": 0.5, "payload": 0.05, "flags": 0.1}

row_bytes = sum(COLUMNS.values()) * ROWS
col_bytes = sum(COLUMNS[c] * ROWS * p for c, p in profile.items())
ratio = col_bytes / row_bytes

print(f"row layout:    {row_bytes:,} bytes")
print(f"column layout: {col_bytes:,.0f} bytes ({ratio:.3f} of row)")

assert row_bytes == 92_000_000
assert round(col_bytes) == 19_600_000
assert round(ratio, 3) == 0.213

### Try it 7. Plan registry subjects for a topic naming scheme

**Task.** For the topics `orders.events.v1` and `payments.ledger.v2`, list the key and value subjects under the TopicName strategy. Sort the result.

**Hint.** A subject is `<topic>-key` or `<topic>-value`. Build the cross product of topics and suffixes, then sort.

**Expected result.** Four subjects: orders.events.v1-key, orders.events.v1-value, payments.ledger.v2-key, payments.ledger.v2-value.

In [ ]:
topics = ["orders.events.v1", "payments.ledger.v2"]
subjects = sorted([f"{topic}-key" for topic in topics]
                  + [f"{topic}-value" for topic in topics])

for subject in subjects:
    print(subject)

assert len(subjects) == 4
assert "orders.events.v1-value" in subjects
assert "payments.ledger.v2-key" in subjects

### Try it 8. Encode and decode a zig-zag varint by hand

**Task.** Implement `zigzag_encode`, `zigzag_decode`, `varint_bytes`, and `varint_decode`. Round-trip several values, including a large positive and negative one, and print the bytes for 8192.

**Hint.** Zig-zag maps signed to unsigned: `(n << 1) ^ (n >> 63)`; decode with `(z >> 1) ^ -(z & 1)`. Base-128 groups 7 bits per byte, and the high bit marks continuation.

**Expected result.** 8192 encodes as the 3 bytes `80 80 01`, and every tested value round-trips through both layers.

In [ ]:
def zigzag_encode(n):
    return (n << 1) ^ (n >> 63)


def zigzag_decode(z):
    return (z >> 1) ^ -(z & 1)


def varint_bytes(z):
    out = bytearray()
    while True:
        byte = z & 0x7F
        z >>= 7
        if z:
            out.append(byte | 0x80)
        else:
            out.append(byte)
            return bytes(out)


def varint_decode(data):
    z = 0
    for i, byte in enumerate(data):
        z |= (byte & 0x7F) << (7 * i)
        if not byte & 0x80:
            return z, i + 1
    raise ValueError("truncated varint")


for n in [0, -1, 1, -64, 64, 8192, -(2**40), 2**40]:
    encoded = varint_bytes(zigzag_encode(n))
    decoded, used = varint_decode(encoded)
    assert zigzag_decode(decoded) == n
    assert used == len(encoded)

print("8192 ->", varint_bytes(zigzag_encode(8192)).hex())
assert varint_bytes(zigzag_encode(8192)) == b"\x80\x80\x01"